In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
from typing import Literal

import numpy as np
import polars as pl
from maite_datasets.object_detection import SeaDrone

from dataeval import Metadata
from dataeval.bias import Balance, Diversity, Parity
from dataeval.data import split_dataset
from dataeval.types import Remap

In [3]:
dataset = SeaDrone("./data", image_set="val", download=True, lazy=True)
raw = Metadata(dataset)
rows = raw.dataframe
frames = rows.filter(pl.col("level") == "unit")

print(
    frames
    .group_by("drone")
    .agg(
        pl.len().alias("frames"),
        (pl.col("altitude") > 0).mean().round(2).alias("altitude recorded"),
        pl.col("altitude").filter(pl.col("altitude") > 0).median().round(0).alias("median altitude (m)"),
        (pl.col("gimbal_pitch") >= 0).mean().round(2).alias("pitch recorded"),
        pl.col("gimbal_pitch").filter(pl.col("gimbal_pitch") >= 0).median().round(0).alias("median pitch (deg)"),
    )
    .sort("frames", descending=True)
)

shape: (3, 6)
┌─────────┬────────┬───────────────────┬─────────────────────┬────────────────┬────────────────────┐
│ drone   ┆ frames ┆ altitude recorded ┆ median altitude (m) ┆ pitch recorded ┆ median pitch (deg) │
│ ---     ┆ ---    ┆ ---               ┆ ---                 ┆ ---            ┆ ---                │
│ str     ┆ u32    ┆ f64               ┆ f64                 ┆ f64            ┆ f64                │
╞═════════╪════════╪═══════════════════╪═════════════════════╪════════════════╪════════════════════╡
│ mavic   ┆ 937    ┆ 1.0               ┆ 37.0                ┆ 1.0            ┆ 36.0               │
│ m210    ┆ 351    ┆ 0.26              ┆ 9.0                 ┆ 0.26           ┆ 54.0               │
│ trinity ┆ 259    ┆ 0.24              ┆ 110.0               ┆ 0.24           ┆ 90.0               │
└─────────┴────────┴───────────────────┴─────────────────────┴────────────────┴────────────────────┘


/tmp/ipykernel_5614/1592517019.py:3: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  rows = raw.dataframe
/tmp/ipykernel_5614/1592517019.py:3: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `object_id`, `object_size`, `speed`, `xspeed`, `yspeed` and `zspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  rows = raw.dataframe


In [4]:
detections = rows.filter(pl.col("level") == "instance").with_columns(
    pl.col("class_label").replace_strict(raw.index2label, return_dtype=pl.String).alias("class")
)
mix = detections.group_by("drone", "class").len().pivot(on="class", index="drone", values="len").fill_null(0)
totals = mix.select(pl.exclude("drone")).sum_horizontal()
print(mix.with_columns((pl.exclude("drone") * 100 / totals).round(0)))

shape: (3, 6)
┌─────────┬─────────┬──────┬──────┬────────┬────────────────────────┐
│ drone   ┆ swimmer ┆ buoy ┆ boat ┆ jetski ┆ life_saving_appliances │
│ ---     ┆ ---     ┆ ---  ┆ ---  ┆ ---    ┆ ---                    │
│ str     ┆ f64     ┆ f64  ┆ f64  ┆ f64    ┆ f64                    │
╞═════════╪═════════╪══════╪══════╪════════╪════════════════════════╡
│ m210    ┆ 93.0    ┆ 0.0  ┆ 5.0  ┆ 2.0    ┆ 0.0                    │
│ trinity ┆ 46.0    ┆ 5.0  ┆ 44.0 ┆ 3.0    ┆ 3.0                    │
│ mavic   ┆ 58.0    ┆ 8.0  ┆ 25.0 ┆ 4.0    ┆ 5.0                    │
└─────────┴─────────┴──────┴──────┴────────┴────────────────────────┘


In [5]:
# TEST ASSERTION CELL ###
by_drone_frames = {
    row["drone"]: row
    for row in frames
    .group_by("drone")
    .agg(
        pl.len().alias("frames"),
        (pl.col("altitude") > 0).mean().alias("recorded"),
        pl.col("altitude").filter(pl.col("altitude") > 0).median().alias("median"),
    )
    .iter_rows(named=True)
}
assert (len(frames), len(detections)) == (1547, 9630)
assert {name: row["frames"] for name, row in by_drone_frames.items()} == {"mavic": 937, "m210": 351, "trinity": 259}
assert by_drone_frames["mavic"]["recorded"] == 1.0
assert all(by_drone_frames[d]["recorded"] < 0.3 for d in ("m210", "trinity"))
assert (
    by_drone_frames["m210"]["median"]
    < 15
    < 30
    < by_drone_frames["mavic"]["median"]
    < 50
    < 100
    < by_drone_frames["trinity"]["median"]
)
shares = {row["drone"]: row for row in mix.iter_rows(named=True)}
swimmer_share = {
    drone: row["swimmer"] / sum(v for k, v in row.items() if k != "drone") for drone, row in shares.items()
}
boat_share = {drone: row["boat"] / sum(v for k, v in row.items() if k != "drone") for drone, row in shares.items()}
assert swimmer_share["m210"] > 0.9
assert 0.4 < swimmer_share["trinity"] < 0.5
assert boat_share["trinity"] > 0.4
assert boat_share["m210"] < 0.1

In [6]:
EXCLUDE = [
    "compass_heading", "frame", "gimbal_heading", "height", "object_id", "speed",
    "storage", "width", "xspeed", "yspeed", "zspeed",
]  # fmt: skip
CORRECTIONS = [Remap("altitude", {(None, 0): None}), Remap("gimbal_pitch", {(None, 0): None})]
ALTITUDE_EDGES = [0, 20, 60, 150, 300]
PITCH_EDGES = [0, 45, 80, 91]

metadata = Metadata(
    dataset,
    exclude=EXCLUDE,
    corrections=CORRECTIONS,
    continuous_factor_bins={"altitude": ALTITUDE_EDGES, "gimbal_pitch": PITCH_EDGES},
    auto_bin_method="uniform_count",
    partial_factors=True,
)

print(f"images:     {metadata.level_counts['unit']}")
print(f"detections: {metadata.level_counts['instance']}")
print(f"factors:    {list(metadata.factor_names)}")
print(f"class axis: {metadata.class_axis} ({metadata.class_axis_source})")

images:     1547
detections: 9630
factors:    ['altitude', 'drone', 'gimbal_pitch', 'object_size']
class axis: class_label (ground_truth)


/tmp/ipykernel_5614/1624747235.py:18: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  print(f"images:     {metadata.level_counts['unit']}")


In [7]:
sizes = detections["object_size"].drop_nulls().to_numpy()
print(f"object size in pixels: median {np.median(sizes):,.0f}, largest {sizes.max():,.0f}")


def quantile_edges(count):
    """Edges that cut the object sizes into bins with the same number of detections."""
    edges = np.quantile(sizes, np.linspace(0, 1, count + 1))
    return [float(edge) for edge in np.unique([0, *edges[1:-1], edges[-1] + 1])]


def associations(bins, method: Literal["uniform_width", "uniform_count"] = "uniform_width", corrections=CORRECTIONS):
    """Score the altitude and pitch against the drone, and the object size against the class, under one binning."""
    candidate = Metadata(
        dataset,
        exclude=EXCLUDE,
        corrections=corrections,
        continuous_factor_bins=bins,
        auto_bin_method=method,
        partial_factors=True,
    )
    by_class = dict(Balance().evaluate(candidate).balance.iter_rows())
    by_platform = dict(Balance().evaluate(candidate.classed_by("drone")).balance.iter_rows())
    return {
        "altitude ~ drone": by_platform["altitude"],
        "pitch ~ drone": by_platform["gimbal_pitch"],
        "size ~ class": by_class["object_size"],
        "drone × altitude cells": len(candidate.at("unit").classed_by("drone", "altitude").index2label),
    }


equal_width = {"altitude": 4, "gimbal_pitch": 4, "object_size": 4}
chosen = {"altitude": ALTITUDE_EDGES, "gimbal_pitch": PITCH_EDGES}
binnings = {
    "Sentinel left in, equal-width bins": associations(equal_width, corrections=None),
    "Sentinel declared missing, equal-width bins": associations(equal_width),
    "Your edges, equal-width object size": associations(chosen | {"object_size": 4}),
    "Your edges, equal-count object size (used)": associations(chosen, "uniform_count"),
    "Coarser edges and bins": associations({
        "altitude": [0, 30, 100, 300],
        "gimbal_pitch": [0, 60, 91],
        "object_size": quantile_edges(3),
    }),
    "Finer edges and bins": associations({
        "altitude": [0, 10, 20, 40, 60, 100, 150, 300],
        "gimbal_pitch": [0, 30, 45, 60, 80, 91],
        "object_size": quantile_edges(8),
    }),
}
with pl.Config(tbl_rows=10, tbl_width_chars=140, fmt_str_lengths=50):
    print(
        pl.DataFrame([{"binning": name, **scores} for name, scores in binnings.items()]).with_columns(
            pl.col(pl.Float64).round(2)
        )
    )

object size in pixels: median 1,700, largest 1,034,176


/tmp/ipykernel_5614/3397317585.py:21: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  by_class = dict(Balance().evaluate(candidate).balance.iter_rows())


/tmp/ipykernel_5614/3397317585.py:21: UserWarning: `object_size` was binned automatically ('uniform_count') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"object_size": [...]} to control this.
  by_class = dict(Balance().evaluate(candidate).balance.iter_rows())


shape: (6, 5)
┌─────────────────────────────────────────────┬──────────────────┬───────────────┬──────────────┬────────────────────────┐
│ binning                                     ┆ altitude ~ drone ┆ pitch ~ drone ┆ size ~ class ┆ drone × altitude cells │
│ ---                                         ┆ ---              ┆ ---           ┆ ---          ┆ ---                    │
│ str                                         ┆ f64              ┆ f64           ┆ f64          ┆ i64                    │
╞═════════════════════════════════════════════╪══════════════════╪═══════════════╪══════════════╪════════════════════════╡
│ Sentinel left in, equal-width bins          ┆ 0.05             ┆ 0.35          ┆ 0.0          ┆ 8                      │
│ Sentinel declared missing, equal-width bins ┆ 0.62             ┆ 0.61          ┆ 0.0          ┆ 9                      │
│ Your edges, equal-width object size         ┆ 0.62             ┆ 0.65          ┆ 0.0          ┆ 11                     │
│ 

In [8]:
# TEST ASSERTION CELL ###
scores = pl.DataFrame([{"binning": name, **row} for name, row in binnings.items()])
left_in, declared, wide, used, coarse, fine = (row for row in scores.iter_rows(named=True))
assert left_in["altitude ~ drone"] < 0.15
assert declared["altitude ~ drone"] > 0.5
assert left_in["pitch ~ drone"] < 0.45 < 0.55 < declared["pitch ~ drone"]
assert wide["size ~ class"] < 0.05
assert 0.25 < used["size ~ class"] < 0.35
assert sizes.max() > 100 * np.median(sizes)
assert all(
    0.55 < row["altitude ~ drone"] < 0.7 and 0.55 < row["pitch ~ drone"] < 0.7
    for row in (declared, wide, used, coarse, fine)
)
assert 0.1 < coarse["size ~ class"] < used["size ~ class"]
assert 0.2 < fine["size ~ class"] < 0.3
assert coarse["drone × altitude cells"] < used["drone × altitude cells"] < fine["drone × altitude cells"]
assert (used["drone × altitude cells"], fine["drone × altitude cells"], coarse["drone × altitude cells"]) == (11, 16, 9)
assert (
    used["altitude ~ drone"] == dict(Balance().evaluate(metadata.classed_by("drone")).balance.iter_rows())["altitude"]
)

/tmp/ipykernel_5614/1889777262.py:19: UserWarning: `object_size` was binned automatically ('uniform_count') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"object_size": [...]} to control this.
  used["altitude ~ drone"] == dict(Balance().evaluate(metadata.classed_by("drone")).balance.iter_rows())["altitude"]


In [9]:
by_drone = metadata.classed_by("drone")

print(f"axis:     {by_drone.class_axis} ({by_drone.class_axis_source})")
print(f"groups:   {dict(by_drone.index2label)}")
print(f"original: {metadata.class_axis} ({metadata.class_axis_source})")

axis:     drone (derived)
groups:   {0: 'm210', 1: 'mavic', 2: 'trinity'}
original: class_label (ground_truth)


In [10]:
print(f"before: {list(metadata.factor_names)}")
print(f"after:  {list(by_drone.factor_names)}")

before: ['altitude', 'drone', 'gimbal_pitch', 'object_size']
after:  ['altitude', 'class', 'gimbal_pitch', 'object_size']


In [11]:
default = Balance().evaluate(metadata)
print(default.balance)

shape: (5, 2)
┌──────────────┬──────────┐
│ factor_name  ┆ mi_value │
│ ---          ┆ ---      │
│ cat          ┆ f64      │
╞══════════════╪══════════╡
│ class_label  ┆ 1.0      │
│ altitude     ┆ 0.042061 │
│ drone        ┆ 0.070877 │
│ gimbal_pitch ┆ 0.056893 │
│ object_size  ┆ 0.303846 │
└──────────────┴──────────┘


In [12]:
result = Balance().evaluate(by_drone)
print(result.balance)

shape: (5, 2)
┌──────────────┬──────────┐
│ factor_name  ┆ mi_value │
│ ---          ┆ ---      │
│ cat          ┆ f64      │
╞══════════════╪══════════╡
│ drone        ┆ 1.0      │
│ altitude     ┆ 0.621857 │
│ class        ┆ 0.085898 │
│ gimbal_pitch ┆ 0.650455 │
│ object_size  ┆ 0.05471  │
└──────────────┴──────────┘


In [13]:
print(default.classwise.filter(pl.col("factor_name") == "altitude"))

shape: (5, 4)
┌────────────────────────┬─────────────┬──────────┬───────────────┐
│ class_name             ┆ factor_name ┆ mi_value ┆ is_imbalanced │
│ ---                    ┆ ---         ┆ ---      ┆ ---           │
│ cat                    ┆ cat         ┆ f64      ┆ bool          │
╞════════════════════════╪═════════════╪══════════╪═══════════════╡
│ boat                   ┆ altitude    ┆ 0.02296  ┆ false         │
│ buoy                   ┆ altitude    ┆ 0.0472   ┆ false         │
│ jetski                 ┆ altitude    ┆ 0.022409 ┆ false         │
│ life_saving_appliances ┆ altitude    ┆ 0.077665 ┆ false         │
│ swimmer                ┆ altitude    ┆ 0.042305 ┆ false         │
└────────────────────────┴─────────────┴──────────┴───────────────┘


In [14]:
print(result.classwise.filter(pl.col("factor_name") == "altitude"))

shape: (3, 4)
┌────────────┬─────────────┬──────────┬───────────────┐
│ class_name ┆ factor_name ┆ mi_value ┆ is_imbalanced │
│ ---        ┆ ---         ┆ ---      ┆ ---           │
│ cat        ┆ cat         ┆ f64      ┆ bool          │
╞════════════╪═════════════╪══════════╪═══════════════╡
│ m210       ┆ altitude    ┆ 0.472167 ┆ true          │
│ mavic      ┆ altitude    ┆ 0.812248 ┆ true          │
│ trinity    ┆ altitude    ┆ 0.351983 ┆ true          │
└────────────┴─────────────┴──────────┴───────────────┘


In [15]:
print(default.classwise.filter(pl.col("factor_name") == "object_size"))

shape: (5, 4)
┌────────────────────────┬─────────────┬──────────┬───────────────┐
│ class_name             ┆ factor_name ┆ mi_value ┆ is_imbalanced │
│ ---                    ┆ ---         ┆ ---      ┆ ---           │
│ cat                    ┆ cat         ┆ f64      ┆ bool          │
╞════════════════════════╪═════════════╪══════════╪═══════════════╡
│ boat                   ┆ object_size ┆ 0.343585 ┆ true          │
│ buoy                   ┆ object_size ┆ 0.31509  ┆ true          │
│ jetski                 ┆ object_size ┆ 0.159284 ┆ false         │
│ life_saving_appliances ┆ object_size ┆ 0.156961 ┆ false         │
│ swimmer                ┆ object_size ┆ 0.259604 ┆ false         │
└────────────────────────┴─────────────┴──────────┴───────────────┘


In [16]:
# TEST ASSERTION CELL ###
assert (metadata.level_counts["unit"], metadata.level_counts["instance"]) == (1547, 9630)
assert sorted(metadata.factor_names) == ["altitude", "drone", "gimbal_pitch", "object_size"]
assert by_drone.index2label == {0: "m210", 1: "mavic", 2: "trinity"}
assert "drone" not in by_drone.factor_names
assert "class" in by_drone.factor_names
by_class = dict(default.balance.iter_rows())
by_platform = dict(result.balance.iter_rows())
assert by_class["object_size"] > 0.25
assert by_class["altitude"] < 0.1
assert by_class["gimbal_pitch"] < 0.1
assert by_platform["altitude"] > 0.5
assert by_platform["gimbal_pitch"] > 0.5
assert by_platform["class"] < 0.15
assert by_platform["object_size"] < 0.1
assert not default.classwise.filter(pl.col("factor_name") == "altitude")["is_imbalanced"].any()
drone_altitude = {
    row["class_name"]: row for row in result.classwise.filter(pl.col("factor_name") == "altitude").iter_rows(named=True)
}
assert all(row["is_imbalanced"] for row in drone_altitude.values())
assert drone_altitude["mavic"]["mi_value"] > 0.7
size_rows = {
    row["class_name"]: row
    for row in default.classwise.filter(pl.col("factor_name") == "object_size").iter_rows(named=True)
}
assert size_rows["boat"]["is_imbalanced"]
assert size_rows["buoy"]["is_imbalanced"]

In [17]:
try:
    Balance().evaluate(metadata.at("unit"))
except ValueError as error:
    print(f"class axis: {error}")

class axis: class_labels is defined at the 'instance' level, but this metadata is viewed at 'unit', which has no label per row. Use md.at('instance') for the labels, or read them from rows_at('unit')["class_label"], or classed_by(...) to condition on a factor these rows do have.


In [18]:
by_drone_per_image = metadata.at("unit").classed_by("drone")
print(Balance().evaluate(by_drone_per_image).balance)

shape: (3, 2)
┌──────────────┬──────────┐
│ factor_name  ┆ mi_value │
│ ---          ┆ ---      │
│ cat          ┆ f64      │
╞══════════════╪══════════╡
│ drone        ┆ 1.0      │
│ altitude     ┆ 0.499972 │
│ gimbal_pitch ┆ 0.558653 │
└──────────────┴──────────┘


In [19]:
print(
    f"detections view: {by_drone.class_labels.shape[0]} rows, fan-out {by_drone.class_axis_info.rows_per_group_entity:.2f}"
)
print(
    f"images view:     {by_drone_per_image.class_labels.shape[0]} rows, "
    f"fan-out {by_drone_per_image.class_axis_info.rows_per_group_entity:.2f}"
)

detections view: 9630 rows, fan-out 6.22
images view:     1547 rows, fan-out 1.00


In [20]:
print(f"detections view factors: {list(by_drone.factor_names)}")
print(f"images view factors:     {list(by_drone_per_image.factor_names)}")

detections view factors: ['altitude', 'class', 'gimbal_pitch', 'object_size']
images view factors:     ['altitude', 'gimbal_pitch']


In [21]:
# TEST ASSERTION CELL ###
unit_balance = dict(Balance().evaluate(by_drone_per_image).balance.iter_rows())
assert unit_balance["altitude"] > 0.4
assert unit_balance["gimbal_pitch"] > 0.4
assert unit_balance["altitude"] < by_platform["altitude"]
assert unit_balance["gimbal_pitch"] < by_platform["gimbal_pitch"]
assert by_drone.class_labels.shape[0] == 9630
assert by_drone_per_image.class_labels.shape[0] == 1547
assert 6.2 < (by_drone.class_axis_info.rows_per_group_entity or 0) < 6.25
assert by_drone_per_image.class_axis_info.rows_per_group_entity == 1.0
assert "class" in by_drone.factor_names
assert "class" not in by_drone_per_image.factor_names

In [22]:
cells = metadata.at("unit").classed_by("drone", "altitude")

print(f"axis: {cells.class_axis}")
counts = np.bincount(cells.class_labels, minlength=len(cells.index2label))
for code, name in sorted(cells.index2label.items(), key=lambda item: item[1]):
    print(f"  {name:24s} {counts[code]:4d} images")

bands = ["[0, 20)", "[20, 60)", "[60, 150)", "[150, 300)", "missing"]
possible = [f"{drone} × {band}" for drone in ("m210", "mavic", "trinity") for band in bands]
absent = [name for name in possible if name not in cells.index2label.values()]
print(f"\ncells present: {len(cells.index2label)} of {len(possible)} possible")
print(f"cells absent:  {absent}")

axis: drone × altitude
  m210 × [0, 20)             56 images
  m210 × [20, 60)            23 images
  m210 × [60, 150)           14 images
  m210 × missing            258 images
  mavic × [0, 20)           201 images
  mavic × [20, 60)          627 images
  mavic × [60, 150)         109 images
  trinity × [150, 300)       19 images
  trinity × [20, 60)         11 images
  trinity × [60, 150)        33 images
  trinity × missing         196 images

cells present: 11 of 15 possible
cells absent:  ['m210 × [150, 300)', 'mavic × [150, 300)', 'mavic × missing', 'trinity × [0, 20)']


In [23]:
# TEST ASSERTION CELL ###
assert len(cells.index2label) == 11
assert len(possible) == 15
assert sorted(absent) == ["m210 × [150, 300)", "mavic × [150, 300)", "mavic × missing", "trinity × [0, 20)"]
assert min(counts) >= 11

In [24]:
recorded = metadata.at("unit").where(pl.col("altitude").is_not_null() & pl.col("gimbal_pitch").is_not_null())
recorded_cells = recorded.classed_by("drone", "altitude")
parity = Parity().evaluate(recorded_cells)
print(parity.factors.select("factor_name", "status", "score", "p_value_adjusted", "has_insufficient_data"))
print(f"\ngrouping: {parity.group_source} ({parity.n_rows} rows, {parity.n_groups} groups, {parity.test} test)")
print(parity.cells["status"].value_counts().sort("status"))

findings = parity.cells.filter(pl.col("is_actionable")).sort("p_value_adjusted").head(8)
print(findings.select("level", "class_name", "status", "observed", "expected", "p_value_adjusted"))

shape: (1, 5)
┌──────────────┬────────────┬──────────┬──────────────────┬───────────────────────┐
│ factor_name  ┆ status     ┆ score    ┆ p_value_adjusted ┆ has_insufficient_data │
│ ---          ┆ ---        ┆ ---      ┆ ---              ┆ ---                   │
│ cat          ┆ enum       ┆ f64      ┆ f64              ┆ bool                  │
╞══════════════╪════════════╪══════════╪══════════════════╪═══════════════════════╡
│ gimbal_pitch ┆ associated ┆ 0.702637 ┆ 1.1695e-222      ┆ true                  │
└──────────────┴────────────┴──────────┴──────────────────┴───────────────────────┘

grouping: rows (1093 rows, 1093 groups, pearson test)
shape: (4, 2)
┌──────────────────┬───────┐
│ status           ┆ count │
│ ---              ┆ ---   │
│ enum             ┆ u32   │
╞══════════════════╪═══════╡
│ underrepresented ┆ 10    │
│ overrepresented  ┆ 3     │
│ too_thin         ┆ 9     │
│ no_finding       ┆ 5     │
└──────────────────┴───────┘
shape: (8, 6)
┌──────────┬─────────────

/tmp/ipykernel_5614/814603991.py:3: ExperimentalWarning: 'Parity' is experimental and may change or be removed in any future release without notice.
  parity = Parity().evaluate(recorded_cells)
/builds/jatic/aria/dataeval/src/dataeval/bias/_parity.py:478: ExperimentalWarning: 'parity' is experimental and may change or be removed in any future release without notice.
  result = parity(factor_data, axis.values, unit.groups, assume_independent=unit.groups is None)
/builds/jatic/aria/dataeval/src/dataeval/bias/_parity.py:601: ExperimentalWarning: 'ParityOutput' is experimental and may change or be removed in any future release without notice.
  return ParityOutput(


In [25]:
# TEST ASSERTION CELL ###
# Lock the claims made in the prose above about the Parity cells.
assert parity.factors["status"].to_list() == ["associated"]
assert 0.6 < parity.factors["score"][0] < 0.8
assert parity.factors["p_value_adjusted"][0] < 1e-6
assert (parity.group_source, parity.test) == ("rows", "pearson")
status_counts = dict(parity.cells["status"].value_counts().iter_rows())
assert status_counts["too_thin"] == 9
assert status_counts["overrepresented"] >= 3
assert status_counts["underrepresented"] >= 8
cell_rows = {(row["class_name"], row["level"]): row for row in parity.cells.iter_rows(named=True)}
assert cell_rows[("m210 × [0, 20)", "[45, 80)")]["status"] == "overrepresented"
assert cell_rows[("mavic × [60, 150)", "[45, 80)")]["status"] == "overrepresented"
assert cell_rows[("mavic × [20, 60)", "[80, 91)")]["status"] == "underrepresented"

In [26]:
diversity = Diversity().evaluate(by_drone)
print(diversity.classwise)

shape: (12, 4)
┌────────────┬──────────────┬─────────────────┬──────────────────┐
│ class_name ┆ factor_name  ┆ diversity_value ┆ is_low_diversity │
│ ---        ┆ ---          ┆ ---             ┆ ---              │
│ cat        ┆ cat          ┆ f64             ┆ bool             │
╞════════════╪══════════════╪═════════════════╪══════════════════╡
│ m210       ┆ altitude     ┆ 0.060378        ┆ true             │
│ m210       ┆ class        ┆ 0.038734        ┆ true             │
│ m210       ┆ gimbal_pitch ┆ 0.0795          ┆ true             │
│ m210       ┆ object_size  ┆ 0.820731        ┆ false            │
│ mavic      ┆ altitude     ┆ 0.186521        ┆ true             │
│ …          ┆ …            ┆ …               ┆ …                │
│ mavic      ┆ object_size  ┆ 0.97243         ┆ false            │
│ trinity    ┆ altitude     ┆ 0.09014         ┆ true             │
│ trinity    ┆ class        ┆ 0.363255        ┆ true             │
│ trinity    ┆ gimbal_pitch ┆ 0.111925        ┆

In [27]:
# TEST ASSERTION CELL ###
div = {(row["class_name"], row["factor_name"]): row for row in diversity.classwise.iter_rows(named=True)}
assert all(div[(d, f)]["is_low_diversity"] for d in ("m210", "mavic", "trinity") for f in ("altitude", "gimbal_pitch"))
assert (
    div[("m210", "class")]["diversity_value"]
    < 0.1
    < 0.3
    < min(div[("mavic", "class")]["diversity_value"], div[("trinity", "class")]["diversity_value"])
)
assert all(
    div[(d, "object_size")]["diversity_value"] > 0.5 and not div[(d, "object_size")]["is_low_diversity"]
    for d in ("m210", "mavic", "trinity")
)

In [28]:
axis = result.class_axis
assert axis
print(f"name:        {axis.name}")
print(f"source:      {axis.source}")
print(f"level:       {axis.level}")
print(f"groups:      {axis.groups}")
print(f"fan-out:     {axis.rows_per_group_entity:.2f} rows per {axis.level}")
print(f"vocabulary:  {axis.vocabulary}")

name:        drone
source:      derived
level:       unit
groups:      3
fan-out:     6.22 rows per unit
vocabulary:  observed


In [29]:
print({k: v for k, v in result.meta().state.items() if k.startswith("class_axis")})

{'class_axis': 'drone', 'class_axis_source': 'derived', 'class_axis_level': 'unit'}


In [30]:
splits = split_dataset(by_drone_per_image, num_folds=2, stratify=True)
assert splits.class_axis
print(f"stratified on: {splits.class_axis.name} ({splits.class_axis.source})")

stratified on: drone (derived)


In [31]:
# TEST ASSERTION CELL ###
assert (axis.name, axis.source, axis.level) == ("drone", "derived", "unit")
assert (splits.class_axis.name, splits.class_axis.source) == ("drone", "derived")